Montgomery County Maryland Wine Sales Dashboard V2

Module imports

In [1]:
import pandas as pd
import numpy as np
import requests
import PyPDF2
import io
import re
import sqlite3
import time
import datetime
import plotly.graph_objects as go
import plotly.express as px
import warnings
import sys
import pickle
import os
import datetime
import streamlit as st
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from difflib import SequenceMatcher
from datetime import datetime
sys.path.append('./utils')
import data_exploration_utils as deu
from wine_classification_utils import run_enhanced_wine_classification, quick_enhanced_summary
exec(open('utils/jupyter_help_instructions.py').read()) #Load help guide for data exploration utility
import inspect
from difflib import SequenceMatcher
from datetime import datetime
from openpyxl import Workbook
from io import StringIO
warnings.filterwarnings('ignore')

Utility functions loaded! Type 'show_help()' for complete guide or 'quick_help()' for shortcuts.


File imports

In [2]:
#GitHub raw URL (not the web interface URL)
base_url = "https://raw.githubusercontent.com/ac604605/Montgomery_County_Dashboard/main/"

# Load Files Directly
try:
    Distributors_Virginia_Three_Main = pd.read_csv(base_url + "data/Distributors_Virginia_Three_Main.csv")
    print(f"Distributors loaded: {Distributors_Virginia_Three_Main.shape}")
    
    wine_producers = pd.read_csv(base_url + "data/wine_producers.csv")
    print(f"Wine Producers loaded: {wine_producers.shape}")

    Warehouse_and_Retail_Sales = pd.read_csv(base_url + "data/Warehouse_and_Retail_Sales.csv")
    print(f"Warehouse and Retail Sales loaded: {Warehouse_and_Retail_Sales.shape}")

    Wine_Review_Data = pd.read_csv(base_url + "data/winemag-data-130k-v2.csv/winemag-data-130k-v2.csv")
    print(f"Wine Review Data loaded: {Wine_Review_Data.shape}")

     # Fix mixed types warning
    Suppliers_Importers_Retailers = pd.read_csv(base_url + "data/Suppliers_Importers_Retailers.csv", 
                                               low_memory=False)
    print(f"Suppliers/Importers/Retailers loaded: {Suppliers_Importers_Retailers.shape}")
    
except Exception as e:
    print(f"Error loading data: {e}")
    print("Check that the file exists in your GitHub repository")

# Display first few rows to verify
print("\nFirst 5 rows of Distributors data:")
print(Distributors_Virginia_Three_Main.head())
print("\nFirst 5 rows of Wine Producer data:")
print(wine_producers.head())
print("\nFirst 5 rows of Warehouse and Retail Sales data:")
print(Warehouse_and_Retail_Sales.head())
print("\nFirst 5 rows of Wine Review Data:")
print(Wine_Review_Data.head())
print("\nFirst 5 rows of Supplier data:")
print(Suppliers_Importers_Retailers.head())

Distributors loaded: (14279, 4)
Wine Producers loaded: (2324, 7)
Warehouse and Retail Sales loaded: (307645, 9)
Wine Review Data loaded: (129971, 14)
Suppliers/Importers/Retailers loaded: (78067, 7)

First 5 rows of Distributors data:
   License_ID             Brand_Name Product_Type Distributor
0       85629              #NICEWINE         Wine        RNDC
1       85629                 10SPAN         Wine        RNDC
2       85629         12 GENERATIONS         Wine        RNDC
3       85629             12 KNIGHTS         Wine        RNDC
4       85629  12 WINES OF CHRISTMAS         Wine        RNDC

First 5 rows of Wine Producer data:
  License ID               Trade Name                   Address        City  \
0     X14533         1026 beverage co           839 S Beacon St   San Pedro   
1     X11997                  117 LLC  1352 Main Street Suite 1  Rutherford   
2     X15111        1413 VALA  AVENUE                       NaN        Napa   
3     X14269  2 X 4 BREWING & IMPORTS   

In [3]:
exec(open('utils/jupyter_help_instructions.py').read()) #Load help guide for data exploration utility

Utility functions loaded! Type 'show_help()' for complete guide or 'quick_help()' for shortcuts.


There was a small error while importing the Supplier data that will be fixed below. 

In [4]:
# Force the correct column structure for Supplier Data
correct_columns = ['License_ID', 'Trade Name', 'Address', 'City', 'State', 'Zip_Code', 'Report_Type']

# Import without header, then assign correct column names
Suppliers_Fixed = pd.read_csv(
    base_url + "data/Suppliers_Importers_Retailers.csv",
    header=0,  # Skip the header row
    names=correct_columns,  # Use our correct column names
    usecols=range(7),  # Only take first 7 columns (ignore trailing comma column)
    dtype={'License_ID': str, 'Zip_Code': str}
)

print("Fixed shape:", Suppliers_Fixed.shape)
print("Fixed columns:", Suppliers_Fixed.columns.tolist())

print("\nSample data:")
print(Suppliers_Fixed.head())

print("\nReport Type distribution:")
print(Suppliers_Fixed['Report_Type'].value_counts().head(10))

Fixed shape: (78067, 7)
Fixed columns: ['License_ID', 'Trade Name', 'Address', 'City', 'State', 'Zip_Code', 'Report_Type']

Sample data:
  License_ID                        Trade Name                   Address  \
0     753721             1480 B C Farm Brewery       10075  Lees Mill RD   
1     752875                2 Silos Brewing Co      9925  Discovery BLVD   
2     095297      50 West Winery and Vineyards     39060  John Mosby HWY   
3     088526  6 Bears & A Goat Brewing Company  1140  International PKWY   
4     132581           600 Vineyard and Winery         14035  Kendall RD   

             City State    Zip_Code                         Report_Type  
0       Warrenton    VA       20186  Beer Shipper - In State (Industry)  
1        Manassas    VA       20109  Beer Shipper - In State (Industry)  
2      Middleburg    VA       20117  Beer Shipper - In State (Industry)  
3  Fredericksburg    VA  22406-1126  Beer Shipper - In State (Industry)  
4          Orange    VA       22960 

Now with everything loaded, I will begin manual exploration of both the Warehouse and Retail Sales data along with the Wine Review Data. The other tables were imported from a reliable source and do not need to be cleaned. First, I will review the sales data. 

Uncomment various lines to review data. The below functions from the accompanying data utilities module are available for further data set analysis. 

In [5]:
help(deu)

Help on module data_exploration_utils:

NAME
    data_exploration_utils

FUNCTIONS
    analyze_duplicates(df: pandas.core.frame.DataFrame, table_name: str, subset: Optional[List[str]] = None, show_examples: bool = True) -> Dict[str, int]
        Analyze duplicate records in the dataset.

        Identifies both complete duplicates and partial duplicates based on
        specified columns. Essential for understanding data quality issues
        and potential data entry problems.

        Args:
            df (pd.DataFrame): DataFrame to analyze for duplicates
            table_name (str): Name for reporting purposes
            subset (list, optional): List of columns to check for partial duplicates.
                                   If None, only checks for complete duplicates.
            show_examples (bool): Whether to show example duplicates (default: True)

        Returns:
            dict: Dictionary containing:
                - 'full_duplicates': Count of complete duplicate r

In [6]:
#uncomment below lines to change dataframe.
#df=Warehouse_and_Retail_Sales
#df=Distributors_Virginia_Three_Main
#df=Wine_Review_Data
#df=Suppliers_Fixed

#uncomment below lines to change report
#filter_column = 'ITEM DESCRIPTION'
#filter_value = 'jackson'
#filtered_df = df[df[filter_column] == filter_value]
#filtered_df = df[df[filter_column].str.contains(filter_value, case=False, na=False)]

#print(f"Found {len(filtered_df)} rows where {filter_column} = '{filter_value}'")
#print(f"Unique SUPPLIER: {filtered_df['SUPPLIER'].nunique()}")
#print(filtered_df.head(25))
#df.head()
#df.info()
#df.describe()
#df.isnull().sum()
#df['ITEM TYPE'].value_counts()
#deu.investigate_missing_values_manually(df)

After investigating the sales data, there are a few cleaning steps that need to take place. First will be removing all values that are not wine and beer items carried by distributors. Second will be ensuring item codes are numeric for easier processesing. Finally, some idividual values will be changed and anything that isn't wine or beer will be removed. I will also be removing the keg versions of wines and beer, as those would introduce greater scope that I do not wish to manage for a simple portfolio. 

Step 1: All null values from the supplier column were found to be items not sold by wine or beer vendors. Mostly store level item sales or distributor credits with no identifying information. All of these values will be removed. 

In [7]:
# Create a copy of the original dataframe
df_clean_Warehouse_and_Retail_Sales = Warehouse_and_Retail_Sales.copy()

print(f"Original shape: {df_clean_Warehouse_and_Retail_Sales.shape}")

# Remove all rows where SUPPLIER is null
df_clean_Warehouse_and_Retail_Sales = df_clean_Warehouse_and_Retail_Sales.dropna(subset=['SUPPLIER'])

print(f"After removing null suppliers: {df_clean_Warehouse_and_Retail_Sales.shape}")
print(f"Rows removed: {df_clean_Warehouse_and_Retail_Sales.shape[0] - df_clean_Warehouse_and_Retail_Sales.shape[0]}")

# Verify no null suppliers remain
print(f"Null suppliers remaining: {df_clean_Warehouse_and_Retail_Sales['SUPPLIER'].isnull().sum()}")

Original shape: (307645, 9)
After removing null suppliers: (307478, 9)
Rows removed: 0
Null suppliers remaining: 0


Step 2: Some item codes have non-numeric values. After investigation, I made a decision to consolidate items with and without non-numberic codes into the same item number. This will help simplify the final graph without adding undue work and with minimal lost data granularity. 

In [8]:
# First, recreate the non_numeric variable
non_numeric = df_clean_Warehouse_and_Retail_Sales[~df_clean_Warehouse_and_Retail_Sales['ITEM CODE'].str.isdigit()]
print(f"Non-numeric item codes found: {len(non_numeric)}")

# Now you can run your pattern analysis
print("Patterns in non-numeric item codes:")
print(non_numeric['ITEM CODE'].str.extract(r'(\d+)([A-Za-z]+)'))

# Count different suffixes
suffixes = non_numeric['ITEM CODE'].str.extract(r'\d+([A-Za-z]+)')
print("\nSuffixes found:")
print(suffixes[0].value_counts())

Non-numeric item codes found: 7
Patterns in non-numeric item codes:
             0  1
48792    97024  A
112864   50029  A
134541  300662  A
138420  348152  A
156556   81130  A
156719   82052  A
172422  239766  A

Suffixes found:
0
A    7
Name: count, dtype: int64


In [9]:
# Look at the patterns in non-numeric codes
print("Patterns in non-numeric item codes:")
print(non_numeric['ITEM CODE'].str.extract(r'(\d+)([A-Za-z]+)'))

# Count different suffixes
suffixes = non_numeric['ITEM CODE'].str.extract(r'\d+([A-Za-z]+)')
print("\nSuffixes found:")
print(suffixes[0].value_counts())

Patterns in non-numeric item codes:
             0  1
48792    97024  A
112864   50029  A
134541  300662  A
138420  348152  A
156556   81130  A
156719   82052  A
172422  239766  A

Suffixes found:
0
A    7
Name: count, dtype: int64


In [10]:
# Check if there are numeric versions of these items
print("Checking for numeric versions of 'A' suffix items...")

for item_code in non_numeric['ITEM CODE'].head(10):
    if item_code.endswith('A'):
        base_code = item_code[:-1]  # Remove the 'A'
        
        # Check if the base code exists
        base_exists = df_clean_Warehouse_and_Retail_Sales['ITEM CODE'].str.contains(f'^{base_code}$').any()
        
        print(f"{item_code} (base: {base_code}) - Base exists: {base_exists}")
        
        if base_exists:
            base_item = df_clean_Warehouse_and_Retail_Sales[df_clean_Warehouse_and_Retail_Sales['ITEM CODE'] == base_code]
            a_item = df_clean_Warehouse_and_Retail_Sales[df_clean_Warehouse_and_Retail_Sales['ITEM CODE'] == item_code]
            
            print(f"  Base: {base_item['ITEM DESCRIPTION'].iloc[0]}")
            print(f"  A version: {a_item['ITEM DESCRIPTION'].iloc[0]}")
            print("---")

Checking for numeric versions of 'A' suffix items...
97024A (base: 97024) - Base exists: True
  Base: GLUTENBERG PALE ALE - 16OZ CAN
  A version: GLUTENBERG PALE ALE - 16OZ CAN
---
50029A (base: 50029) - Base exists: True
  Base: AVERY ELLIES BROWN 4/6 12OZ CANS
  A version: AVERY ELLIES BROWN 4/6 12OZ CAN
---
300662A (base: 300662) - Base exists: True
  Base: DR STONERS SMOKY HERB WHISKEY 750ML
  A version: DR STONERS SMOKY HERB WHISKEY 750ML
---
348152A (base: 348152) - Base exists: True
  Base: VIRGINIA BLACK WHISKEY - 750ML
  A version: VIRGINIA BLACK WHISKEY - 750ML
---
81130A (base: 81130) - Base exists: True
  Base: CH LEOGNAN ROUGE 15 - 750ML
  A version: CH LEOGNAN '15 - 750ML
---
82052A (base: 82052) - Base exists: True
  Base: CH RIEUSSEC '14 - 750ML
  A version: CH RIEUSSEC '14 -750ML
---
239766A (base: 239766) - Base exists: True
  Base: EMPORIUM APPASSIMENTO SALENTO - 750ML
  A version: EMPORIUM APPASSIMENTO SALENTO - 750ML
---


In [11]:
# Step 1: Identify all A-suffix items that have base versions
a_suffix_items = df_clean_Warehouse_and_Retail_Sales[df_clean_Warehouse_and_Retail_Sales['ITEM CODE'].str.endswith('A')].copy()
a_suffix_items['BASE_CODE'] = a_suffix_items['ITEM CODE'].str[:-1]  # Remove the 'A'

print(f"Found {len(a_suffix_items)} items with 'A' suffix")

# Step 2: Check which ones have corresponding base versions
base_codes_exist = a_suffix_items['BASE_CODE'].isin(df_clean_Warehouse_and_Retail_Sales['ITEM CODE'])
consolidatable = a_suffix_items[base_codes_exist].copy()

print(f"Can consolidate {len(consolidatable)} items (have matching base codes)")

# Step 3: For each A-suffix item, copy the base version's key columns
for idx, row in consolidatable.iterrows():
    base_code = row['BASE_CODE']
    a_code = row['ITEM CODE']
    
    # Find the base version
    base_row = df_clean_Warehouse_and_Retail_Sales[df_clean_Warehouse_and_Retail_Sales['ITEM CODE'] == base_code].iloc[0]
    
    # Update the A-version to match base version for key columns
    df_clean_Warehouse_and_Retail_Sales.loc[df_clean_Warehouse_and_Retail_Sales['ITEM CODE'] == a_code, 'SUPPLIER'] = base_row['SUPPLIER']
    df_clean_Warehouse_and_Retail_Sales.loc[df_clean_Warehouse_and_Retail_Sales['ITEM CODE'] == a_code, 'ITEM CODE'] = base_row['ITEM CODE'] 
    df_clean_Warehouse_and_Retail_Sales.loc[df_clean_Warehouse_and_Retail_Sales['ITEM CODE'] == a_code, 'ITEM DESCRIPTION'] = base_row['ITEM DESCRIPTION']
    df_clean_Warehouse_and_Retail_Sales.loc[df_clean_Warehouse_and_Retail_Sales['ITEM CODE'] == a_code, 'ITEM TYPE'] = base_row['ITEM TYPE']

print("Consolidation complete!")

# Step 4: Verify the changes
print("\nVerification - should now see duplicates:")
sample_base = consolidatable['BASE_CODE'].iloc[0]
consolidated_items = df_clean_Warehouse_and_Retail_Sales[df_clean_Warehouse_and_Retail_Sales['ITEM CODE'] == sample_base]
print(f"Items with code {sample_base}: {len(consolidated_items)}")
print(consolidated_items[['ITEM CODE', 'ITEM DESCRIPTION']].head())

Found 7 items with 'A' suffix
Can consolidate 7 items (have matching base codes)
Consolidation complete!

Verification - should now see duplicates:
Items with code 97024: 22
      ITEM CODE                ITEM DESCRIPTION
11871     97024  GLUTENBERG PALE ALE - 16OZ CAN
23097     97024  GLUTENBERG PALE ALE - 16OZ CAN
35116     97024  GLUTENBERG PALE ALE - 16OZ CAN
48791     97024  GLUTENBERG PALE ALE - 16OZ CAN
48792     97024  GLUTENBERG PALE ALE - 16OZ CAN


In [12]:
# Find ALL non-numeric item codes in your clean dataset
print("Finding all non-numeric item codes...")

# Check which ones are NOT purely numeric
non_numeric = df_clean_Warehouse_and_Retail_Sales[~df_clean_Warehouse_and_Retail_Sales['ITEM CODE'].str.isdigit()]
print(f"Non-numeric item codes found: {len(non_numeric)}")

if len(non_numeric) > 0:
    print("\nUnique non-numeric item codes:")
    print(non_numeric['ITEM CODE'].value_counts())
    
    print("\nSample of these items:")
    print(non_numeric[['ITEM CODE', 'ITEM DESCRIPTION', 'ITEM TYPE']].head(10))

Finding all non-numeric item codes...
Non-numeric item codes found: 0


In [13]:
# Convert to numeric
df_clean_Warehouse_and_Retail_Sales['ITEM CODE'] = pd.to_numeric(df_clean_Warehouse_and_Retail_Sales['ITEM CODE'])

# Verify the conversion
print(f"\nAfter conversion:")
print(f"New ITEM CODE dtype: {df_clean_Warehouse_and_Retail_Sales['ITEM CODE'].dtype}")
print(f"Sample item codes: {df_clean_Warehouse_and_Retail_Sales['ITEM CODE'].head().tolist()}")
print(f"Any NaN values created: {df_clean_Warehouse_and_Retail_Sales['ITEM CODE'].isnull().sum()}")


After conversion:
New ITEM CODE dtype: int64
Sample item codes: [100009, 100024, 1001, 100145, 100293]
Any NaN values created: 0


Step 3: After investigating the various item types available in the report, I have decided to remove all except wine and beer. This will lose some sales from wine and beer kegs, but it will simplify the report. Again this is being done for simplicity. 

In [14]:
# Check current counts
print("Before filtering:")
print(df_clean_Warehouse_and_Retail_Sales['ITEM TYPE'].value_counts())
print(f"Total rows: {len(df_clean_Warehouse_and_Retail_Sales)}")

# Filter to keep only WINE and BEER
df_clean_Warehouse_and_Retail_Sales = df_clean_Warehouse_and_Retail_Sales[df_clean_Warehouse_and_Retail_Sales['ITEM TYPE'].isin(['WINE', 'BEER'])]

# Check results
print("\nAfter filtering:")
print(df_clean_Warehouse_and_Retail_Sales['ITEM TYPE'].value_counts())
print(f"Total rows: {len(df_clean_Warehouse_and_Retail_Sales)}")

# Calculate what was removed
wine_beer_total = 187640 + 42413
print(f"\nRows kept: {len(df_clean_Warehouse_and_Retail_Sales)} (WINE + BEER)")
print(f"Rows removed: {307478 - len(df_clean_Warehouse_and_Retail_Sales)} (LIQUOR, KEGS, etc.)")

Before filtering:
ITEM TYPE
WINE            187640
LIQUOR           64910
BEER             42413
KEGS             10146
NON-ALCOHOL       1899
STR_SUPPLIES       318
REF                 79
DUNNAGE             72
Name: count, dtype: int64
Total rows: 307478

After filtering:
ITEM TYPE
WINE    187640
BEER     42413
Name: count, dtype: int64
Total rows: 230053

Rows kept: 230053 (WINE + BEER)
Rows removed: 77425 (LIQUOR, KEGS, etc.)


I will also perform some joins and various cleaning of supporting data tables meant to make brand ownership rights more clear. 

In [15]:
"""
Fuzzy Supplier Matching System
Handles name variations, case differences, and business suffix inconsistencies
"""



def clean_supplier_name(name):
    """
    Clean supplier names for better matching by removing common variations
    """
    if pd.isna(name):
        return ""
    
    name = str(name).upper().strip()
    
    # Remove common business suffixes and variations
    suffixes_to_remove = [
        ' INC', ' LLC', ' CO', ' CORP', ' CORPORATION', ' LTD', ' LIMITED', 
        ' COMPANY', ' LP', ' LLP', ' LLLP', ' DBA', ' USA', ' INC.', ' LLC.', 
        ' CO.', ' CORP.', ' LTD.', ' LP.', ' LLP.'
    ]
    
    for suffix in suffixes_to_remove:
        if name.endswith(suffix):
            name = name[:-len(suffix)].strip()
    
    # Remove extra spaces and punctuation
    name = ' '.join(name.split())  # Normalize whitespace
    name = name.replace('.', '').replace(',', '').replace('&', 'AND')
    
    return name

def find_best_supplier_match(sales_supplier, suppliers_df, threshold=0.8):
    """
    Find the best matching supplier using fuzzy string matching
    
    Args:
        sales_supplier (str): Supplier name from sales data
        suppliers_df (pd.DataFrame): Suppliers reference table
        threshold (float): Minimum similarity score (0.8 = 80% similar)
    
    Returns:
        tuple: (best_match_name, similarity_score, report_type)
    """
    sales_clean = clean_supplier_name(sales_supplier)
    
    if not sales_clean:
        return None, 0, None
    
    best_match = None
    best_score = 0
    best_report_type = None
    
    for _, row in suppliers_df.iterrows():
        supplier_name = row['Trade Name']
        supplier_clean = clean_supplier_name(supplier_name)
        
        if not supplier_clean:
            continue
        
        # Calculate similarity score
        score = SequenceMatcher(None, sales_clean, supplier_clean).ratio()
        
        # Bonus points for exact word matches
        sales_words = set(sales_clean.split())
        supplier_words = set(supplier_clean.split())
        
        if sales_words and supplier_words:
            word_overlap = len(sales_words.intersection(supplier_words)) / len(sales_words.union(supplier_words))
            score = (score * 0.7) + (word_overlap * 0.3)  # Weighted combination
        
        if score > best_score and score >= threshold:
            best_score = score
            best_match = supplier_name
            best_report_type = row.get('Report_Type', None)
    
    return best_match, best_score, best_report_type

def enrich_sales_with_supplier_data(sales_df, suppliers_df, threshold=0.8, 
                                   show_progress=True, max_test_rows=None):
    """
    Enrich sales data with supplier information using fuzzy matching
    
    Args:
        sales_df (pd.DataFrame): Sales data with 'SUPPLIER' column
        suppliers_df (pd.DataFrame): Suppliers data with 'Trade Name' and 'Report Type'
        threshold (float): Minimum similarity threshold for matching
        show_progress (bool): Whether to show progress updates
        max_test_rows (int): Limit rows for testing (None = process all)
    
    Returns:
        pd.DataFrame: Enriched sales data with new columns
    """
    print(f"Starting fuzzy supplier matching with threshold {threshold}")
    print(f"Sales data: {len(sales_df):,} rows")
    
    # Filter suppliers to only Wholesale Wine Distributors
    wine_distributors = suppliers_df[suppliers_df['Report_Type'] == 'Wholesale Wine Distributors']
    print(f"Wholesale Wine Distributors: {len(wine_distributors):,} (filtered from {len(suppliers_df):,} total suppliers)")
    
    # Create working copy
    df_enriched = sales_df.copy()
    
    # For testing, limit rows
    if max_test_rows:
        df_enriched = df_enriched.head(max_test_rows)
        print(f"Testing mode: Processing only {max_test_rows:,} rows")
    
    # Add new columns
    df_enriched['MATCHED_SUPPLIER_NAME'] = ""
    df_enriched['SUPPLIER_MATCH_SCORE'] = 0.0
    df_enriched['SUPPLIER_REPORT_TYPE'] = ""
    
    # Get unique suppliers to avoid duplicate work
    unique_suppliers = df_enriched['SUPPLIER'].unique()
    print(f"Unique suppliers to match: {len(unique_suppliers):,}")
    
    # Create supplier lookup cache
    supplier_cache = {}
    
    start_time = time.time()
    matched_count = 0
    
    # Process each unique supplier
    for i, supplier in enumerate(unique_suppliers):
        if pd.isna(supplier):
            continue
            
        # Find best match (only among wine distributors)
        match_name, match_score, report_type = find_best_supplier_match(
            supplier, wine_distributors, threshold
        )
        
        # Cache the result
        supplier_cache[supplier] = {
            'matched_name': match_name or "",
            'match_score': match_score,
            'report_type': 'Wholesale Wine Distributors' if match_name else ""
        }
        
        if match_name:
            matched_count += 1
        
        # Progress reporting
        if show_progress and (i + 1) % 50 == 0:
            elapsed = time.time() - start_time
            print(f"Processed {i+1:,}/{len(unique_suppliers):,} suppliers "
                  f"({matched_count} matches, {elapsed:.1f}s)")
    
    # Apply cached results to all rows
    print("Applying matches to all rows...")
    for supplier, cache_data in supplier_cache.items():
        mask = df_enriched['SUPPLIER'] == supplier
        df_enriched.loc[mask, 'MATCHED_SUPPLIER_NAME'] = cache_data['matched_name']
        df_enriched.loc[mask, 'SUPPLIER_MATCH_SCORE'] = cache_data['match_score']
        df_enriched.loc[mask, 'SUPPLIER_REPORT_TYPE'] = cache_data['report_type']
    
    # Final statistics
    total_time = time.time() - start_time
    total_matched_rows = (df_enriched['SUPPLIER_MATCH_SCORE'] >= threshold).sum()
    match_rate = (total_matched_rows / len(df_enriched)) * 100
    
    print(f"\n{'='*50}")
    print("FUZZY MATCHING RESULTS:")
    print(f"Unique suppliers matched: {matched_count:,}/{len(unique_suppliers):,}")
    print(f"Total rows matched: {total_matched_rows:,}/{len(df_enriched):,} ({match_rate:.1f}%)")
    print(f"Processing time: {total_time:.1f} seconds")
    print(f"Average time per supplier: {total_time/len(unique_suppliers):.3f}s")
    
    return df_enriched

def test_supplier_matching(sales_df, suppliers_df, test_suppliers=None):
    """
    Test the matching function on specific suppliers
    """
    # Filter to only Wholesale Wine Distributors for testing
    wine_distributors = suppliers_df[suppliers_df['Report_Type'] == 'Wholesale Wine Distributors']
    print(f"Testing against {len(wine_distributors):,} Wholesale Wine Distributors")
    
    if test_suppliers is None:
        # Use some common suppliers for testing
        test_suppliers = [
            "REPUBLIC NATIONAL DISTRIBUTING CO",
            "SANTA MARGHERITA USA INC", 
            "SUTTER HOME WINERY INC",
            "JACKSON FAMILY ENTERPRISES INC"
        ]
    
    print("Testing supplier matching:")
    print("="*50)
    
    for supplier in test_suppliers:
        match_name, score, report_type = find_best_supplier_match(
            supplier, wine_distributors, threshold=0.7
        )
        
        print(f"Sales: '{supplier}'")
        print(f"Match: '{match_name}' (score: {score:.3f})")
        print(f"Type:  '{report_type}'")
        print("-" * 30)

# Usage functions
def run_supplier_enrichment(sales_df, suppliers_df, test_mode=False):
    """
    Main function to run supplier enrichment
    """
    if test_mode:
        print("Running in TEST MODE with 1000 rows...")
        result = enrich_sales_with_supplier_data(
            sales_df, suppliers_df, 
            threshold=0.8, 
            max_test_rows=1000
        )
    else:
        print("Running FULL enrichment...")
        result = enrich_sales_with_supplier_data(
            sales_df, suppliers_df, 
            threshold=0.8
        )
    
    return result

def analyze_supplier_matches(enriched_df):
    """
    Analyze the results of supplier matching
    """
    print("SUPPLIER MATCHING ANALYSIS")
    print("="*40)
    
    # Match rate analysis
    matched = enriched_df[enriched_df['SUPPLIER_MATCH_SCORE'] >= 0.8]
    match_rate = len(matched) / len(enriched_df) * 100
    print(f"Overall match rate: {match_rate:.1f}%")
    
    # Score distribution
    print(f"\nMatch score distribution:")
    print(enriched_df['SUPPLIER_MATCH_SCORE'].describe())
    
    # Report types found
    print(f"\nSupplier Report Types found:")
    print(enriched_df['SUPPLIER_REPORT_TYPE'].value_counts())
    
    # Sample matches
    print(f"\nSample successful matches:")
    sample_matches = matched[['SUPPLIER', 'MATCHED_SUPPLIER_NAME', 
                             'SUPPLIER_MATCH_SCORE', 'SUPPLIER_REPORT_TYPE']].head(10)
    print(sample_matches)
    
    return matched

In [16]:
df_clean_Warehouse_and_Retail_Sales_enhanced = run_supplier_enrichment(df_clean_Warehouse_and_Retail_Sales, 
                                     Suppliers_Fixed, test_mode=False)

Running FULL enrichment...
Starting fuzzy supplier matching with threshold 0.8
Sales data: 230,053 rows
Wholesale Wine Distributors: 440 (filtered from 78,067 total suppliers)
Unique suppliers to match: 340
Processed 50/340 suppliers (11 matches, 3.8s)
Processed 100/340 suppliers (13 matches, 7.6s)
Processed 150/340 suppliers (17 matches, 11.2s)
Processed 200/340 suppliers (19 matches, 15.1s)
Processed 250/340 suppliers (23 matches, 19.0s)
Processed 300/340 suppliers (24 matches, 22.9s)
Applying matches to all rows...

FUZZY MATCHING RESULTS:
Unique suppliers matched: 24/340
Total rows matched: 48,657/230,053 (21.2%)
Processing time: 32.6 seconds
Average time per supplier: 0.096s


In [17]:
# Check which suppliers got matched
matched_suppliers = df_clean_Warehouse_and_Retail_Sales_enhanced[
    (df_clean_Warehouse_and_Retail_Sales_enhanced['SUPPLIER_MATCH_SCORE'] >= 0.8) & 
    (df_clean_Warehouse_and_Retail_Sales_enhanced['SUPPLIER_REPORT_TYPE'] == 'Wholesale Wine Distributors')
]

print("Your Wholesale Wine Distributors:")
print(matched_suppliers['SUPPLIER'].value_counts())

print("\nSample matched data:")
print(matched_suppliers[['SUPPLIER', 'MATCHED_SUPPLIER_NAME', 'SUPPLIER_MATCH_SCORE']].head(10))

Your Wholesale Wine Distributors:
SUPPLIER
REPUBLIC NATIONAL DISTRIBUTING CO        17598
MONSIEUR TOUTON SELECTION                10326
DIONYSOS IMPORTS INC                      4120
KYSELA PERE ET FILS LTD                   3733
ELITE WINES IMPORTS                       3042
BARON FRANCOIS LTD                        2065
GRAPES OF SPAIN INC                       1562
ARTISANS & VINES LLC                      1206
IMPERO WINE DISTRIBUTORS VIRGINIA INC      893
VIGNOBLES LVDH USA INC                     778
INTERNATIONAL CELLARS LLC                  599
MICHAEL R DOWNEY SELECTIONS INC            593
TRADEWINDS SPECIALTY IMPORTS LLC           592
FIVE GRAPES LLC                            465
FREE RUN WINE MERCHANTS LLC                448
NICKOLAS IMPORTS LLC                       163
POTOMAC SELECTIONS INC                     142
CANTINIERE IMPORTS & DISTRIBUTING INC      134
IMPERO WINE DISTRIBUTORS                    61
WILLIAMS CORNER WINE                        50
Z WINE GALLERY IM

Now I will begin manual exploration of both the Wine Review Data. 

Uncomment various lines to review data. The below functions from the accompanying data utilities module are available for further data set analysis. 

In [18]:
#uncomment below lines to change dataframe.
#df=Warehouse_and_Retail_Sales
#df=Distributors_Virginia_Three_Main
#df=Wine_Review_Data
#df=df_clean_Warehouse_and_Retail_Sales_enhanced
#df=Suppliers_Fixed

#uncomment below lines to change report
#df.head()
#df.info()
#df.describe()
#df.isnull().sum()
#df['country'].value_counts()
#deu.investigate_missing_values_manually(df)

After reviewing the wine review data, it appears this will serve as an adequate database to gather missing country data for our sales table. For the sake of simplicity, I will combine all columns from this table to matching wines in our sales table. Columns we do not need can be filtered out later. 

Step 1: Testing sample search terms on the wine review data. 

In [19]:
search_terms = ["SANTA MARGHERITA", "ROBERT MONDAVI", "CUPCAKE", "CAKEBREAD", "BUTTER"]

for term in search_terms:
    print(f"\n--- Searching for '{term}' ---")
    matches = Wine_Review_Data[Wine_Review_Data['title'].str.contains(term, case=False, na=False)]
    print(f"Found {len(matches)} matches")
    if len(matches) > 0:
        print("Sample matches:")
        for idx, row in matches.head(3).iterrows():
            print(f"  - {row['title']} | Country: {row['country']}")


--- Searching for 'SANTA MARGHERITA' ---
Found 10 matches
Sample matches:
  - Panizzi 2014 Vigna Santa Margherita  (Vernaccia di San Gimignano) | Country: Italy
  - Panizzi 2006 Vigna Santa Margherita  (Vernaccia di San Gimignano) | Country: Italy
  - Santa Margherita 2015 Extra Dry  (Valdobbiadene Prosecco Superiore) | Country: Italy

--- Searching for 'ROBERT MONDAVI' ---
Found 157 matches
Sample matches:
  - Robert Mondavi 2015 Fumé Blanc (Napa Valley) | Country: US
  - Robert Mondavi 2008 Cabernet Sauvignon (Napa Valley) | Country: US
  - Robert Mondavi 2011 Reserve Chardonnay (Carneros) | Country: US

--- Searching for 'CUPCAKE' ---
Found 31 matches
Sample matches:
  - Cupcake 2016 Rosé (California) | Country: US
  - Cupcake 2010 Cabernet Sauvignon (Central Coast) | Country: US
  - Cupcake 2012 Petite Sirah (Central Coast) | Country: US

--- Searching for 'CAKEBREAD' ---
Found 15 matches
Sample matches:
  - Cakebread 2014 Two Creeks Vineyards Pinot Noir (Anderson Valley) | Countr

After testing simple search results above, I will begin to create a search algorithm below. First up, testing a simple scoring function to differentiate between wines with similar names. 

In [20]:
def score_match(search_term, wine_title):
    """Score how well a search term matches a wine title."""
    
    # Convert to uppercase for case-insensitive comparison
    search_term = search_term.upper()
    wine_title = wine_title.upper()
    
    score = 0
    # Check if it's an exact producer match
    if wine_title.startswith(search_term):
        score += 100
    
    return score

def find_best_matches(search_term, Wine_Review_Data, top_n=5):
    """
    Find the best matching wines for a search term and return them with scores
    """
    matches = []
    
    # First, find all wines that contain our search term
    potential_matches = Wine_Review_Data[Wine_Review_Data['title'].str.contains(search_term, case=False, na=False, regex=False)]
    
    # Now score each potential match
    for index, row in potential_matches.iterrows():
        score = score_match(search_term, row['title'])
        matches.append({
            'title': row['title'],
            'country': row['country'],
            'score': score
        })
    
    # Sort by score (highest first) and return top matches
    matches = sorted(matches, key=lambda x: x['score'], reverse=True)
    return matches[:top_n]

# Test the functions work together
print("Testing complete matching system:")
print("=" * 40)

test_terms = ["SANTA MARGHERITA", "ROBERT MONDAVI", "CUPCAKE"]
for term in test_terms:
    results = find_best_matches(term, Wine_Review_Data, top_n=3)
    print(f"\nTop matches for {term}:")
    for match in results:
        print(f"  Score: {match['score']} | {match['title']} | Country: {match['country']}")

Testing complete matching system:

Top matches for SANTA MARGHERITA:
  Score: 100 | Santa Margherita 2015 Extra Dry  (Valdobbiadene Prosecco Superiore) | Country: Italy
  Score: 100 | Santa Margherita 2007 Pinot Grigio (Alto Adige) | Country: Italy
  Score: 100 | Santa Margherita 2002 Pinot Grigio (Valdadige) | Country: Italy

Top matches for ROBERT MONDAVI:
  Score: 100 | Robert Mondavi 2015 Fumé Blanc (Napa Valley) | Country: US
  Score: 100 | Robert Mondavi 2008 Cabernet Sauvignon (Napa Valley) | Country: US
  Score: 100 | Robert Mondavi 2011 Reserve Chardonnay (Carneros) | Country: US

Top matches for CUPCAKE:
  Score: 100 | Cupcake 2016 Rosé (California) | Country: US
  Score: 100 | Cupcake 2010 Cabernet Sauvignon (Central Coast) | Country: US
  Score: 100 | Cupcake 2012 Petite Sirah (Central Coast) | Country: US


In [21]:
def add_all_match_columns_with_backup(df_clean_Warehouse_and_Retail_Sales, Wine_Review_Data, resume_from_checkpoint=None):
    """
    Enhanced version with better backup and resume capability
    """
    import time
    import datetime
    import pickle
    import os
    
    print("Starting country/enrichment matching with enhanced backup...")
    
    # Resume from checkpoint if provided
    if resume_from_checkpoint and os.path.exists(resume_from_checkpoint):
        print(f"🔄 Resuming from checkpoint: {resume_from_checkpoint}")
        df_with_matches = pd.read_pickle(resume_from_checkpoint)
        
        # Find where to resume
        wine_mask = df_with_matches['ITEM TYPE'] == 'WINE'
        processed_wines = df_with_matches[wine_mask & (df_with_matches['PRODUCER_FOUND'] != '')]
        start_idx = len(processed_wines)
        print(f"📍 Resuming from wine #{start_idx:,}")
    else:
        df_with_matches = df_clean_Warehouse_and_Retail_Sales.copy()
        start_idx = 0
        
        # Initialize columns (only if starting fresh)
        Wine_Review_Data_columns = Wine_Review_Data.columns.tolist()
        print(f"Will add {len(Wine_Review_Data_columns)} columns from Wine_Review_Data dataset")
        
        for col in Wine_Review_Data_columns:
            if col in df_with_matches.columns:
                new_col_name = f"Wine_Review_Data_{col}"
            else:
                new_col_name = col
            
            if col.lower() in ['country']:
                df_with_matches[new_col_name] = 'Unknown'
            elif col.lower() in ['points', 'price', 'score', 'rating']:
                df_with_matches[new_col_name] = 0
            else:
                df_with_matches[new_col_name] = ''
        
        # Initialize tracking columns
        df_with_matches['PRODUCER_FOUND'] = ''
        df_with_matches['MATCH_CONFIDENCE'] = 0
        df_with_matches['TOTAL_MATCHES'] = 0
    
    description_cache = {}
    matched_count = len(df_with_matches[df_with_matches['PRODUCER_FOUND'] != ''])
    cache_hits = 0
    
    wine_mask = df_with_matches['ITEM TYPE'] == 'WINE'
    wine_items = df_with_matches[wine_mask].iloc[start_idx:]  # Resume from checkpoint
    total_wines = len(df_with_matches[wine_mask])
    
    print(f"Processing {len(wine_items)} remaining wines (of {total_wines} total)...")
    
    start_time = time.time()
    
    for i, (idx, row) in enumerate(wine_items.iterrows()):
        current_position = start_idx + i
        desc = row['ITEM DESCRIPTION']
        
        # Skip if already processed
        if row['PRODUCER_FOUND'] != '':
            continue
        
        # Check cache first
        if desc in description_cache:
            result = description_cache[desc]
            cache_hits += 1
        else:
            # Extract search term from description (adjust logic as needed)
            search_term = ' '.join(desc.split()[:2])  # First 2 words
            matches = find_best_matches(search_term, Wine_Review_Data, top_n=1)
    
            if matches and len(matches) > 0:  # This line needs to be indented
                # Get the full row from Wine_Review_Data for the best match
                best_match = matches[0]
                matched_rows = Wine_Review_Data[Wine_Review_Data['title'] == best_match['title']]
                
                if len(matched_rows) > 0:
                    matched_row_dict = matched_rows.iloc[0].to_dict()
                    result = {
                        'producer_found': search_term,
                        'confidence': best_match['score'],
                        'total_matches': len(matches),
                        'matched_row': matched_row_dict
                    }
                else:
                    result = None
            else:
                result = None
    
            description_cache[desc] = result
            
        if result:  # This line should align with the 'if desc in description_cache:' above
            # Add Wine_Review_Data columns
            if 'matched_row' in result and result['matched_row'] is not None:
                matched_row = result['matched_row']
                Wine_Review_Data_columns = Wine_Review_Data.columns.tolist()
                
                for col in Wine_Review_Data_columns:
                    if col in df_clean_Warehouse_and_Retail_Sales.columns:
                        new_col_name = f"Wine_Review_Data_{col}"
                    else:
                        new_col_name = col
                    
                    value = matched_row.get(col, '')
                    df_with_matches.at[idx, new_col_name] = value
            
            # Update tracking columns
            df_with_matches.at[idx, 'PRODUCER_FOUND'] = result.get('producer_found', '')
            df_with_matches.at[idx, 'MATCH_CONFIDENCE'] = result.get('confidence', 0)
            df_with_matches.at[idx, 'TOTAL_MATCHES'] = result.get('total_matches', 0)
            matched_count += 1
        
        # Save checkpoint every 500 items (more frequent)
        if (current_position + 1) % 500 == 0:
            checkpoint_filename = f'wine_matching_checkpoint_{current_position+1}.pkl'
            df_with_matches.to_pickle(checkpoint_filename)
            print(f"*** 💾 CHECKPOINT SAVED: {checkpoint_filename} ***")
        
        # Progress reporting every 100 items
        if (current_position + 1) % 100 == 0:
            elapsed_time = time.time() - start_time
            processed = current_position + 1
            
            if processed > start_idx:
                avg_time = elapsed_time / (processed - start_idx)
                remaining = total_wines - processed
                eta_seconds = remaining * avg_time
                eta = str(datetime.timedelta(seconds=int(eta_seconds)))
            else:
                eta = "Calculating..."
            
            print(f"Progress: {processed:,}/{total_wines:,} ({processed/total_wines*100:.1f}%)")
            print(f"  Matches: {matched_count:,} | ETA: {eta}")
    
    # Final save
    final_filename = f'wine_matching_FINAL_{datetime.datetime.now().strftime("%Y%m%d_%H%M")}.pkl'
    df_with_matches.to_pickle(final_filename)
    print(f"🎉 FINAL RESULTS SAVED: {final_filename}")
    
    return df_with_matches

In [22]:
# Start fresh
#result = add_all_match_columns_with_backup(df_clean_Warehouse_and_Retail_Sales, Wine_Review_Data)

# Or resume from a checkpoint
#result = add_all_match_columns_with_backup(df_clean, Wine_Review_Data, resume_from_checkpoint='wine_matching_checkpoint_50000.pkl')

After investigating the resulting dataframe, there seem to be some matching inefficiencies. I'm going to work on identifying and improving those below. First, I'll review all table data and check that column names and table names are matched correctly. There were some discrepancies between naming conventions from various sites. 

In [23]:
"""
Improved Wine Review Matching System with Integrated Column Validation
Matches sales data item descriptions directly against wine review titles
"""

import pandas as pd
import time
from datetime import datetime, timedelta
import re
from difflib import SequenceMatcher
from typing import Dict, List, Optional, Tuple
import warnings

class ColumnValidator:
    """Handle column validation and mapping for wine matching system"""
    
    def __init__(self):
        # Define expected column mappings with alternatives
        self.sales_columns = {
            'item_type': ['ITEM TYPE', 'ITEM_TYPE', 'ItemType', 'item_type'],
            'item_description': ['ITEM DESCRIPTION', 'ITEM_DESCRIPTION', 'ItemDescription', 'item_description'],
            'supplier': ['SUPPLIER', 'supplier'],
            'item_code': ['ITEM CODE', 'ITEM_CODE', 'ItemCode', 'item_code'],
            'year': ['YEAR', 'year'],
            'month': ['MONTH', 'month'],
        }
        
        self.review_columns = {
            'title': ['title', 'Title', 'TITLE', 'wine_title'],
            'country': ['country', 'Country', 'COUNTRY'],
            'variety': ['variety', 'Variety', 'VARIETY'],
            'points': ['points', 'Points', 'POINTS', 'rating', 'score'],
            'price': ['price', 'Price', 'PRICE'],
            'description': ['description', 'Description', 'DESCRIPTION'],
            'province': ['province', 'Province', 'PROVINCE', 'region'],
            'region_1': ['region_1', 'Region_1', 'REGION_1', 'region1'],
            'region_2': ['region_2', 'Region_2', 'REGION_2', 'region2'],
            'winery': ['winery', 'Winery', 'WINERY'],
            'designation': ['designation', 'Designation', 'DESIGNATION'],
            'taster_name': ['taster_name', 'Taster_Name', 'TASTER_NAME', 'taster'],
        }
    
    def find_column(self, df: pd.DataFrame, column_key: str, column_dict: Dict) -> Optional[str]:
        """Find the actual column name in the DataFrame"""
        possible_names = column_dict.get(column_key, [])
        
        for name in possible_names:
            if name in df.columns:
                return name
        return None
    
    def validate_and_map_columns(self, sales_df: pd.DataFrame, 
                                review_df: pd.DataFrame) -> Tuple[Dict[str, str], Dict[str, str]]:
        """
        Validate required columns exist and create mapping dictionaries
        
        Returns:
            Tuple of (sales_column_map, review_column_map)
        """
        # Required columns for basic functionality
        required_sales = ['item_type', 'item_description']
        required_reviews = ['title']
        
        # Create column mappings
        sales_map = {}
        review_map = {}
        
        # Validate sales columns
        missing_sales = []
        for col_key in required_sales:
            actual_col = self.find_column(sales_df, col_key, self.sales_columns)
            if actual_col:
                sales_map[col_key] = actual_col
            else:
                missing_sales.append(col_key)
        
        # Validate review columns
        missing_reviews = []
        for col_key in required_reviews:
            actual_col = self.find_column(review_df, col_key, self.review_columns)
            if actual_col:
                review_map[col_key] = actual_col
            else:
                missing_reviews.append(col_key)
        
        # Add optional columns that exist
        optional_sales = ['supplier', 'item_code', 'year', 'month']
        for col_key in optional_sales:
            actual_col = self.find_column(sales_df, col_key, self.sales_columns)
            if actual_col:
                sales_map[col_key] = actual_col
        
        optional_reviews = ['country', 'variety', 'points', 'price', 'description', 
                          'province', 'region_1', 'region_2', 'winery', 'designation', 'taster_name']
        for col_key in optional_reviews:
            actual_col = self.find_column(review_df, col_key, self.review_columns)
            if actual_col:
                review_map[col_key] = actual_col
        
        # Report issues
        if missing_sales or missing_reviews:
            error_msg = "Missing required columns:\n"
            if missing_sales:
                available_sales = list(sales_df.columns)
                error_msg += f"Sales data missing: {missing_sales}\n"
                error_msg += f"Available sales columns: {available_sales}\n"
            if missing_reviews:
                available_reviews = list(review_df.columns)
                error_msg += f"Review data missing: {missing_reviews}\n"
                error_msg += f"Available review columns: {available_reviews}\n"
            raise ValueError(error_msg)
        
        return sales_map, review_map
    
    def print_column_mapping(self, sales_map: Dict[str, str], review_map: Dict[str, str]):
        """Print the column mapping for verification"""
        print("✅ Column Mapping Detected:")
        print("Sales Data:")
        for key, actual in sales_map.items():
            print(f"  {key} -> '{actual}'")
        
        print("Review Data:")
        for key, actual in review_map.items():
            print(f"  {key} -> '{actual}'")
        print()


def clean_text_for_matching(text):
    """Clean text for better matching"""
    if pd.isna(text):
        return ""
    
    text = str(text).upper()
    
    # Remove common wine suffixes and volume indicators
    text = re.sub(r'\s*-\s*(750ML|1\.5L|375ML|187ML|3L|500ML|1L)\s*$', '', text)
    text = re.sub(r'\s*-\s*$', '', text)  # Remove trailing dashes
    
    # Remove common business suffixes
    suffixes_to_remove = [
        'INC', 'LLC', 'CO', 'CORP', 'CORPORATION', 'LTD', 'LIMITED',
        'COMPANY', 'WINERY', 'VINEYARDS', 'VINEYARD', 'WINES', 'WINE'
    ]
    
    for suffix in suffixes_to_remove:
        text = re.sub(rf'\b{suffix}\b', '', text)
    
    # Clean up extra spaces and punctuation
    text = re.sub(r'[^\w\s]', ' ', text)
    text = ' '.join(text.split())
    
    return text.strip()


def extract_wine_name_from_description(item_description):
    """Extract searchable wine name from item description"""
    desc = item_description.upper().strip()
    
    # Remove volume first
    desc = re.sub(r'\s*-\s*(750ML|1\.5L|375ML|187ML|3L|500ML|1L)\s*$', '', desc)
    
    # Handle common abbreviations
    abbreviation_map = {
        'CH ': 'CHATEAU ',
        'DOM ': 'DOMAINE ',
        'S/BLC': 'SAUVIGNON BLANC',
        'P/GRIG': 'PINOT GRIGIO', 
        'P/GRIS': 'PINOT GRIS',
        'S/BLANC': 'SAUVIGNON BLANC',
        'P/NOIR': 'PINOT NOIR',
        'CAB SAV': 'CABERNET SAUVIGNON',
        'CAB': 'CABERNET',
        'CHARD': 'CHARDONNAY'
    }
    
    for abbrev, full_name in abbreviation_map.items():
        desc = desc.replace(abbrev, full_name)
    
    return desc.strip()


def find_best_wine_review_match(wine_name, wine_review_data, review_map, threshold=0.6):
    """
    Find the best matching wine review using two-stage approach
    """
    wine_clean = clean_text_for_matching(wine_name)
    if not wine_clean or len(wine_clean) < 3:
        return None, 0
    
    # Stage 1: Fast pre-filtering using substring search
    # Split wine name into key words and search for any of them
    search_words = wine_clean.split()
    if not search_words:
        return None, 0
    
    # Create search pattern - look for wines containing any key words
    search_pattern = '|'.join([word for word in search_words if len(word) > 2])
    
    if not search_pattern:
        return None, 0
    
    # Use the mapped column name for title
    title_col = review_map['title']
    
    try:
        potential_matches = wine_review_data[
            wine_review_data[title_col].str.contains(search_pattern, case=False, na=False, regex=True)
        ]
    except (re.error, ValueError) as e:
        # Fallback to simple contains if regex fails
        potential_matches = wine_review_data[
            wine_review_data[title_col].str.contains(search_words[0], case=False, na=False, regex=False)
        ]
    
    if len(potential_matches) == 0:
        return None, 0
    
    # Stage 2: Fuzzy matching on filtered candidates
    best_match = None
    best_score = 0
    
    # Limit candidates for performance (top 50 pre-filtered results)
    candidates = potential_matches.head(50)
    
    for _, wine_row in candidates.iterrows():
        candidate_title = clean_text_for_matching(wine_row[title_col])
        if not candidate_title:
            continue
        
        # Calculate similarity score
        similarity = SequenceMatcher(None, wine_clean, candidate_title).ratio()
        
        # Bonus for word matches
        wine_words = set(wine_clean.split())
        candidate_words = set(candidate_title.split())
        
        if wine_words and candidate_words:
            word_overlap = len(wine_words.intersection(candidate_words)) / len(wine_words)
            similarity += word_overlap * 0.3  # 30% bonus for word matches
        
        if similarity > best_score and similarity >= threshold:
            best_score = similarity
            best_match = wine_row
    
    return best_match, best_score


def match_wines_to_reviews(sales_df, wine_review_data, threshold=0.6, 
                          max_test_rows=None, show_progress=True):
    """
    Match wine sales data to wine review data with integrated column validation
    
    Args:
        sales_df: DataFrame with wine sales data
        wine_review_data: DataFrame with wine review data
        threshold: Minimum similarity score for matching (0.6 = 60%)
        max_test_rows: Limit for testing (None = process all)
        show_progress: Whether to show progress updates
    
    Returns:
        DataFrame with wine review data added, plus column mappings
    """
    print(f"🍷 Starting Wine Review Matching with Validation...")
    
    # Step 1: Validate and map columns
    validator = ColumnValidator()
    try:
        sales_map, review_map = validator.validate_and_map_columns(sales_df, wine_review_data)
        validator.print_column_mapping(sales_map, review_map)
    except ValueError as e:
        print(f"❌ Column validation failed: {e}")
        return None, None, None
    
    print(f"Sales data: {len(sales_df):,} rows")
    print(f"Wine reviews: {len(wine_review_data):,} rows")
    print(f"Matching threshold: {threshold}")
    
    # Step 2: Filter to wines using the mapped column name
    wine_mask = sales_df[sales_map['item_type']] == 'WINE'
    wine_sales = sales_df[wine_mask].copy()
    
    if max_test_rows:
        wine_sales = wine_sales.head(max_test_rows)
        print(f"Testing mode: Processing {len(wine_sales):,} wines")
    else:
        print(f"Processing {len(wine_sales):,} wines")
    
    # Step 3: Initialize new columns for wine review data
    # Add review columns with consistent naming, skipping index columns
    for col_key, actual_col in review_map.items():
        if actual_col in ['Unnamed: 0']:  # Skip index columns
            continue
        new_col_name = f'review_{col_key}'
        if new_col_name not in wine_sales.columns:
            wine_sales[new_col_name] = ""
    
    # Add metadata columns
    wine_sales['WINE_NAME_EXTRACTED'] = ""
    wine_sales['REVIEW_MATCH_SCORE'] = 0.0
    wine_sales['REVIEW_MATCH_STATUS'] = 'NO_MATCH'
    
    # Step 4: Caching for performance
    match_cache = {}
    matched_count = 0
    cache_hits = 0
    
    start_time = time.time()
    
    # Step 5: Process each wine
    item_desc_col = sales_map['item_description']
    
    for i, (idx, row) in enumerate(wine_sales.iterrows()):
        item_desc = row[item_desc_col]
        
        # Check cache first
        if item_desc in match_cache:
            result = match_cache[item_desc]
            cache_hits += 1
        else:
            # Extract wine name and find match
            wine_name = extract_wine_name_from_description(item_desc)
            wine_match, match_score = find_best_wine_review_match(
                wine_name, wine_review_data, review_map, threshold
            )
            
            result = {
                'wine_name': wine_name,
                'match_data': wine_match,
                'score': match_score
            }
            match_cache[item_desc] = result
        
        # Apply results
        wine_sales.at[idx, 'WINE_NAME_EXTRACTED'] = result['wine_name']
        wine_sales.at[idx, 'REVIEW_MATCH_SCORE'] = result['score']
        
        if result['match_data'] is not None:
            wine_sales.at[idx, 'REVIEW_MATCH_STATUS'] = 'MATCHED'
            matched_count += 1
            
            # Add all review columns using the mapped names
            for col_key, actual_col in review_map.items():
                if actual_col in ['Unnamed: 0']:
                    continue
                review_col_name = f'review_{col_key}'
                if review_col_name in wine_sales.columns:
                    wine_sales.at[idx, review_col_name] = result['match_data'][actual_col]
        
        # Progress reporting
        if show_progress and (i + 1) % 1000 == 0:
            elapsed = time.time() - start_time
            processed = i + 1
            avg_time = elapsed / processed
            remaining = len(wine_sales) - processed
            eta = str(timedelta(seconds=int(remaining * avg_time)))
            
            match_rate = (matched_count / processed) * 100
            cache_rate = (cache_hits / processed) * 100
            
            print(f"Progress: {processed:,}/{len(wine_sales):,} ({processed/len(wine_sales)*100:.1f}%)")
            print(f"  Matches: {matched_count:,} ({match_rate:.1f}%)")
            print(f"  Cache hits: {cache_rate:.1f}%")
            print(f"  ETA: {eta}")
    
    # Final statistics
    total_time = time.time() - start_time
    final_match_rate = (matched_count / len(wine_sales)) * 100
    
    print(f"\n{'='*50}")
    print("🍷 WINE REVIEW MATCHING RESULTS:")
    print(f"Wines processed: {len(wine_sales):,}")
    print(f"Matches found: {matched_count:,} ({final_match_rate:.1f}%)")
    print(f"Cache efficiency: {(cache_hits/len(wine_sales)*100):.1f}%")
    print(f"Processing time: {str(timedelta(seconds=int(total_time)))}")
    
    return wine_sales, sales_map, review_map


def run_wine_review_matching(sales_df, wine_review_data, threshold=0.6, test_mode=False):
    """
    Main function to run wine review matching with validation
    
    Returns:
        Tuple of (results_df, sales_column_map, review_column_map)
    """
    if test_mode:
        print("🧪 Running in TEST MODE with 1000 wines...")
        return match_wines_to_reviews(
            sales_df, wine_review_data, 
            threshold=threshold,
            max_test_rows=1000
        )
    else:
        print("🚀 Running FULL wine review matching...")
        return match_wines_to_reviews(
            sales_df, wine_review_data,
            threshold=threshold
        )


def safe_column_access(df, column_map, key):
    """
    Safe way to access columns using the mapping
    
    Usage: 
        item_desc = safe_column_access(sales_df, sales_map, 'item_description')
    """
    if key not in column_map:
        raise KeyError(f"Column key '{key}' not found in mapping")
    
    actual_column = column_map[key]
    if actual_column not in df.columns:
        raise KeyError(f"Mapped column '{actual_column}' not found in DataFrame")
    
    return df[actual_column]


# Example usage:
"""
# Test first with small sample
test_results, sales_map, review_map = run_wine_review_matching(
    df_clean_Warehouse_and_Retail_Sales_enhanced, 
    Wine_Review_Data, 
    threshold=0.6,
    test_mode=True
)

if test_results is not None:
    # Check results
    print("\\nSample matches:")
    matched = test_results[test_results['REVIEW_MATCH_STATUS'] == 'MATCHED']
    sample_cols = ['ITEM DESCRIPTION', 'WINE_NAME_EXTRACTED', 'REVIEW_MATCH_SCORE', 
                  'review_country', 'review_variety', 'review_points']
    # Only show columns that exist
    available_cols = [col for col in sample_cols if col in matched.columns]
    print(matched[available_cols].head())

    # If results look good, run on full dataset
    print("\\n" + "="*50)
    print("Results look good! Running full dataset...")
    
    full_results, sales_map, review_map = run_wine_review_matching(
        df_clean_Warehouse_and_Retail_Sales_enhanced,
        Wine_Review_Data,
        threshold=0.6,
        test_mode=False
    )
"""

'\n# Test first with small sample\ntest_results, sales_map, review_map = run_wine_review_matching(\n    df_clean_Warehouse_and_Retail_Sales_enhanced, \n    Wine_Review_Data, \n    threshold=0.6,\n    test_mode=True\n)\n\nif test_results is not None:\n    # Check results\n    print("\\nSample matches:")\n    matched = test_results[test_results[\'REVIEW_MATCH_STATUS\'] == \'MATCHED\']\n    sample_cols = [\'ITEM DESCRIPTION\', \'WINE_NAME_EXTRACTED\', \'REVIEW_MATCH_SCORE\', \n                  \'review_country\', \'review_variety\', \'review_points\']\n    # Only show columns that exist\n    available_cols = [col for col in sample_cols if col in matched.columns]\n    print(matched[available_cols].head())\n\n    # If results look good, run on full dataset\n    print("\\n" + "="*50)\n    print("Results look good! Running full dataset...")\n    \n    full_results, sales_map, review_map = run_wine_review_matching(\n        df_clean_Warehouse_and_Retail_Sales_enhanced,\n        Wine_Review_Da

In [24]:
full_results = run_wine_review_matching(
    df_clean_Warehouse_and_Retail_Sales_enhanced,
    Wine_Review_Data,
    threshold=0.6,
    test_mode=False
)

🚀 Running FULL wine review matching...
🍷 Starting Wine Review Matching with Validation...
✅ Column Mapping Detected:
Sales Data:
  item_type -> 'ITEM TYPE'
  item_description -> 'ITEM DESCRIPTION'
  supplier -> 'SUPPLIER'
  item_code -> 'ITEM CODE'
  year -> 'YEAR'
  month -> 'MONTH'
Review Data:
  title -> 'title'
  country -> 'country'
  variety -> 'variety'
  points -> 'points'
  price -> 'price'
  description -> 'description'
  province -> 'province'
  region_1 -> 'region_1'
  region_2 -> 'region_2'
  winery -> 'winery'
  designation -> 'designation'
  taster_name -> 'taster_name'

Sales data: 230,053 rows
Wine reviews: 129,971 rows
Matching threshold: 0.6
Processing 187,640 wines


KeyboardInterrupt: 

In [ ]:
# Save as pickle (recommended for data analysis)
#full_results.to_pickle('wine_sales_with_reviews_FINAL.pkl')

# To load later:
df = pd.read_pickle('wine_sales_with_reviews_FINAL.pkl')

In [ ]:
#uncomment below lines to change dataframe.
#df=Warehouse_and_Retail_Sales
#df=Distributors_Virginia_Three_Main
#df=Wine_Review_Data
#df=df_clean_Warehouse_and_Retail_Sales_enhanced
#df=Suppliers_Fixed
df=df

#uncomment below lines to change report
#df.head()
df.info()
#df.describe()
#df.isnull().sum()
#df['review_designation'].value_counts()
#deu.investigate_missing_values_manually(df)

After revieweing the table, our data enrichment has gone well but many of the wines from the review data do not follow standard naming conventions for our industry. I will create a secondary classification system to help simplify future steps. This classification system has been loaded as a utility

In [ ]:
#df_final, variety_counts = run_complete_wine_classification(df)

In [ ]:
# See full documentation for the main function
#help(run_complete_wine_classification)

# See documentation for the class
#help(CompleteWineClassificationSystem)

# See documentation for specific methods
#help(CompleteWineClassificationSystem.process_complete_classification)

In [ ]:
#uncomment below lines to change dataframe.
#df=Warehouse_and_Retail_Sales
#df=Distributors_Virginia_Three_Main
#df=Wine_Review_Data
#df=df_clean_Warehouse_and_Retail_Sales_enhanced
#df=Suppliers_Fixed
#df=df_loaded
#df=df_final
#df=variety_counts

#uncomment below lines to change report
#df.head(25)
df.info()
#df.describe()
#df[df['review_country'] == ''].head()
df.head()
#pd.set_option('display.max_rows', None)
#df['ITEM DESCRIPTION'].value_counts()
#deu.investigate_missing_values_manually(df)

In [ ]:
def classify_missing_variety_enhanced(item_description):
    """
    Enhanced variety extraction from item descriptions
    """
    if not item_description:
        return 'Unknown'
    
    desc = item_description.upper()
    
    # Direct variety matches (original)
    if 'GRIGIO' in desc or 'P/GRIGIO' in desc:
        return 'Pinot Grigio'
    elif 'CHARDONNAY' in desc or 'CHARD' in desc:
        return 'Chardonnay'
    elif 'CABERNET' in desc:
        if 'SAUVIGNON' in desc:
            return 'Cabernet Sauvignon'
        else:
            return 'Cabernet Sauvignon'
    elif 'MERLOT' in desc:
        return 'Merlot'
    elif 'SAUVIGNON' in desc and 'CABERNET' not in desc:
        return 'Sauvignon Blanc'
    elif 'PINOT' in desc and 'NOIR' in desc:
        return 'Pinot Noir'
    elif 'PINOT' in desc:
        return 'Pinot Grigio'
    elif 'SYRAH' in desc:
        return 'Syrah'
    elif 'RIESLING' in desc:
        return 'Riesling'
    elif 'MOSCATO' in desc:
        return 'Moscato'
    
    # NEW: More variety patterns I spotted
    elif 'MALB' in desc:  # "HOUSE MALB", "FAIRTRADE MALB"
        return 'Malbec'
    elif 'TORRONTES' in desc:
        return 'Torrontés'
    elif 'GRUNER VELT' in desc or 'GRÜNER VELT' in desc:
        return 'Grüner Veltliner'
    elif 'ALBR' in desc or 'ALBARIÑO' in desc:  # "PAZO DAS BRUXAS ALBR"
        return 'Albariño'
    elif 'PRIMITIVO' in desc:
        return 'Primitivo'
    elif 'SAPERAVI' in desc:
        return 'Saperavi'
    elif 'VIDAL' in desc:
        return 'Vidal Blanc'
    elif 'PICPOUL' in desc:
        return 'Picpoul'
    elif 'NEGRA' in desc:  # "MAS DONIS NEGRA"
        return 'Red Blend'
    
    # Regional/style indicators (original + new)
    elif 'POUILLY FUME' in desc:
        return 'Sauvignon Blanc'
    elif 'MONTEPULCIANO' in desc or 'MONTEPUL' in desc or 'MONT/PUL' in desc:
        return 'Montepulciano'
    elif 'CARMENERE' in desc:
        return 'Carmenère'
    elif 'SPARK' in desc or 'SPUMANTE' in desc:
        return 'Sparkling Blend'
    elif 'VERMENTINO' in desc:
        return 'Vermentino'
    
    # NEW: Regional indicators
    elif 'ST EMILION' in desc:
        return 'Red Blend'  # Bordeaux blend
    elif 'SUPER TUSCAN' in desc:
        return 'Red Blend'
    elif 'PROVENCE' in desc and ('RSE' in desc or 'ROSE' in desc):
        return 'Rosé'
    elif 'PECORION' in desc:  # "PECORION SABELLI"
        return 'Pecorino'
    elif 'ICE' in desc and 'VIDAL' in desc:
        return 'Ice Wine'
    
    # Special categories
    elif 'VERMOUTH' in desc or 'VERM' in desc:
        return 'Vermouth'
    elif 'MEAD' in desc:
        return 'Mead'
    elif 'SAKE' in desc or any(x in desc for x in ['TOZAI', 'KOOK SOON']):
        return 'Sake'
    elif 'MD 20/20' in desc:
        return 'Fortified Wine'
    elif 'COLD DUCK' in desc:
        return 'Sparkling Blend'
    elif 'MANGRIA' in desc:
        return 'Fruit Wine'
    
    # Color-based fallbacks
    elif any(word in desc for word in ['RED', 'ROUGE', 'ROSSO', 'TINTO', 'RGE']):
        return 'Red Blend'
    elif any(word in desc for word in ['WHITE', 'BLANC', 'BLANCO', 'BIANCO']):
        return 'White Blend'
    elif any(word in desc for word in ['ROSE', 'RSE', 'ROSÉ']):
        return 'Rosé'
    
    else:
        return 'Unknown'

# Test the enhanced function
enhanced_results = missing_varieties['ITEM DESCRIPTION'].apply(classify_missing_variety_enhanced)
print("Enhanced classification results:")
print(enhanced_results.value_counts())
print(f"\nTotal classifiable: {(enhanced_results != 'Unknown').sum()}")
print(f"Still unknown: {(enhanced_results == 'Unknown').sum()}")
print(f"Success rate: {(enhanced_results != 'Unknown').sum() / len(enhanced_results) * 100:.1f}%")

In [ ]:
# Get the unknowns from enhanced classification
enhanced_results = missing_varieties['ITEM DESCRIPTION'].apply(classify_missing_variety_enhanced)
still_unknown = missing_varieties[enhanced_results == 'Unknown']

# Get additional context columns
unknown_with_context = still_unknown[['ITEM DESCRIPTION', 'SUPPLIER', 'review_variety', 
                                     'review_variety_consolidated', 'extracted_variety', 
                                     'review_country', 'review_winery', 'WINE_NAME_EXTRACTED']].copy()

# Sort by item description frequency
unknown_counts = still_unknown['ITEM DESCRIPTION'].value_counts()

print("Top unknown item descriptions with additional context:")
print("=" * 100)

for i, (description, count) in enumerate(unknown_counts.head(30).items(), 1):
    print(f"\n{i:2d}. ({count:2d}x) {description}")
    
    # Get a sample record for this description to show context
    sample = unknown_with_context[unknown_with_context['ITEM DESCRIPTION'] == description].iloc[0]
    
    # Only show non-empty additional info
    context_info = []
    if pd.notna(sample['SUPPLIER']) and sample['SUPPLIER'].strip():
        context_info.append(f"Supplier: {sample['SUPPLIER']}")
    if pd.notna(sample['review_variety']) and sample['review_variety'].strip():
        context_info.append(f"Review Variety: {sample['review_variety']}")
    if pd.notna(sample['review_country']) and sample['review_country'].strip():
        context_info.append(f"Country: {sample['review_country']}")
    if pd.notna(sample['review_winery']) and sample['review_winery'].strip():
        context_info.append(f"Winery: {sample['review_winery']}")
    if pd.notna(sample['WINE_NAME_EXTRACTED']) and sample['WINE_NAME_EXTRACTED'].strip():
        context_info.append(f"Wine Name: {sample['WINE_NAME_EXTRACTED']}")
    
    if context_info:
        print(f"    Context: {' | '.join(context_info)}")
    else:
        print(f"    Context: No additional info available")

print(f"\nTotal unique unknown descriptions: {len(unknown_counts)}")
print(f"Total unknown records: {unknown_counts.sum()}")

In [ ]:
# Check existing variety categories in your full dataset
print("Current variety categories in your database:")
all_varieties = df['final_variety'].value_counts()
print(f"Total unique varieties: {len(all_varieties)}")
print("\nTop 50 varieties:")
print(all_varieties.head(50))

# Check for specific patterns that might help with edge cases
print("\n" + "="*50)
print("Checking for specific patterns in existing data:")

# Check for fruit wine patterns
fruit_patterns = all_varieties[all_varieties.index.str.contains('Fruit|Berry|Peach|Pomegranate', case=False, na=False)]
if len(fruit_patterns) > 0:
    print(f"\nFruit wine patterns found:")
    print(fruit_patterns)
else:
    print("\nNo fruit wine patterns found in existing varieties")

# Check for Italian blend patterns
italian_patterns = all_varieties[all_varieties.index.str.contains('Italian|Tuscany|Chianti', case=False, na=False)]
if len(italian_patterns) > 0:
    print(f"\nItalian wine patterns found:")
    print(italian_patterns)
else:
    print("\nNo specific Italian patterns found")

# Check for moscato patterns
moscato_patterns = all_varieties[all_varieties.index.str.contains('Moscato|Muscat', case=False, na=False)]
if len(moscato_patterns) > 0:
    print(f"\nMoscato patterns found:")
    print(moscato_patterns)
else:
    print("\nNo Moscato variations found")

In [ ]:
def classify_missing_variety_final(item_description):
    """
    Final enhanced variety extraction from item descriptions
    """
    if not item_description:
        return 'Unknown'
    
    desc = item_description.upper()
    
    # Direct variety matches
    if 'GRIGIO' in desc or 'P/GRIGIO' in desc:
        return 'Pinot Grigio'
    elif 'CHARDONNAY' in desc or 'CHARD' in desc:
        return 'Chardonnay'
    elif 'CABERNET' in desc:
        if 'SAUVIGNON' in desc:
            return 'Cabernet Sauvignon'
        else:
            return 'Cabernet Sauvignon'
    elif 'MERLOT' in desc:
        return 'Merlot'
    elif 'SAUVIGNON' in desc and 'CABERNET' not in desc:
        return 'Sauvignon Blanc'
    elif 'PINOT' in desc and 'NOIR' in desc:
        return 'Pinot Noir'
    elif 'P/NERO' in desc:  # Italian Pinot Nero
        return 'Pinot Noir'
    elif 'PINOT' in desc:
        return 'Pinot Grigio'
    elif 'SYRAH' in desc:
        return 'Syrah'
    elif 'RIESLING' in desc or 'RIE' in desc:  # Added RIE for German abbreviations
        return 'Riesling'
    elif 'MOSCATO' in desc:
        return 'Moscato'
    elif 'TEMPRANILLO' in desc:
        return 'Tempranillo'
    
    # More variety patterns
    elif 'MALB' in desc:
        return 'Malbec'
    elif 'TORRONTES' in desc:
        return 'Torrontés'
    elif 'GRUNER VELT' in desc or 'GRÜNER VELT' in desc:
        return 'Grüner Veltliner'
    elif 'ALBR' in desc or 'ALBARIÑO' in desc:
        return 'Albariño'
    elif 'PRIMITIVO' in desc:
        return 'Primitivo'
    elif 'SAPERAVI' in desc:
        return 'Saperavi'
    elif 'VIDAL' in desc:
        return 'Vidal Blanc'
    elif 'PICPOUL' in desc:
        return 'Picpoul'
    elif 'NEGRA' in desc:
        return 'Red Blend'
    
    # Regional/style indicators
    elif 'POUILLY FUME' in desc:
        return 'Sauvignon Blanc'
    elif 'MONTEPULCIANO' in desc or 'MONTEPUL' in desc or 'MONT/PUL' in desc:
        return 'Montepulciano'
    elif 'CARMENERE' in desc:
        return 'Carmenère'
    elif 'VERMENTINO' in desc:
        return 'Vermentino'
    elif 'ST EMILION' in desc:
        return 'Red Blend'
    elif 'SUPER TUSCAN' in desc:
        return 'Red Blend'
    elif 'PROVENCE' in desc and ('RSE' in desc or 'ROSE' in desc):
        return 'Rosé'
    elif 'PECORION' in desc:
        return 'Pecorino'
    elif 'ICE' in desc and 'VIDAL' in desc:
        return 'Ice Wine'
    elif 'VALPOLICELLA' in desc or 'RIPASSA VAL' in desc:
        return 'Valpolicella'
    elif 'GSM' in desc:  # Grenache/Syrah/Mourvèdre
        return 'Red Blend'
    
    # NEW: Specific wine name patterns from your list
    elif 'JAZZ BERRY' in desc:
        return 'Fruit Wine'
    elif 'APOTHIC' in desc and 'DARK' in desc:
        return 'Red Blend'
    elif 'CARLO ROSSI PAISANO' in desc:
        return 'Red Blend'
    elif 'SWEET WALTER WHT' in desc:
        return 'White Blend'
    elif 'CHOCOVINE' in desc:
        return 'Red Blend'
    elif 'GOLLY WOBBLER' in desc:
        return 'Red Blend'
    elif 'PECHE IMPERIALE' in desc:
        return 'Moscato'
    elif 'ORIN SWIFT' in desc:
        return 'Red Blend'
    elif 'ANCIANO' in desc and 'RES' in desc:
        return 'Tempranillo'
    elif 'POMEGRANTE' in desc or 'POMEGRANATE' in desc:
        return 'Fruit Wine'
    elif 'PALAZZO DEL TORRE' in desc:
        return 'Red Blend'
    elif 'RES DUCALE' in desc:  # Ruffino Chianti
        return 'Chianti'
    elif 'SEY/CH/VID' in desc:  # Seyval/Chardonnay/Vidal blend
        return 'White Blend'
    
    # Sparkling wines
    elif any(x in desc for x in ['SPARK', 'SPUMANTE', 'ASTI', 'CHAMPAGNE', 
                                 'BL DE NOIR', 'BLANC DE NOIRS', 'CORDON NEGRO',
                                 'ACE OF SPADE']):
        return 'Sparkling Blend'
    elif 'PROSECCO' in desc:
        return 'Prosecco'
    elif 'FREIXENET' in desc:
        return 'Sparkling Blend'
    
    # Fortified wines
    elif any(x in desc for x in ['VERMOUTH', 'VERM', 'SHERRY', 'PORT', 'MADEIRA',
                                 'MARSALA', 'RAINWATER', 'BRISTOL CREAM']):
        return 'Vermouth' if 'VERM' in desc else 'Sherry' if any(x in desc for x in ['SHERRY', 'BRISTOL CREAM']) else 'Madeira' if any(x in desc for x in ['MADEIRA', 'RAINWATER']) else 'Marsala' if 'MARSALA' in desc else 'Port'
    
    # Special categories
    elif 'MEAD' in desc:
        return 'Mead'
    elif 'SAKE' in desc or any(x in desc for x in ['TOZAI', 'KOOK SOON']):
        return 'Sake'
    elif any(x in desc for x in ['MAKKOLI', 'MAKGEOLLI']):
        return 'Sake'  # Korean rice wine, but closest category
    elif 'MD 20/20' in desc:
        return 'Fortified Wine'
    elif 'COLD DUCK' in desc:
        return 'Sparkling Blend'
    elif any(x in desc for x in ['PLUM', 'CHOYA']):
        return 'Fruit Wine'
    
    # Color-based fallbacks
    elif any(word in desc for word in ['RED', 'ROUGE', 'ROSSO', 'TINTO', 'RGE']):
        return 'Red Blend'
    elif any(word in desc for word in ['WHITE', 'BLANC', 'BLANCO', 'BIANCO']):
        return 'White Blend'
    elif any(word in desc for word in ['ROSE', 'RSE', 'ROSÉ']):
        return 'Rosé'

# Quick additions to the classification function:

    # Easy sparkling wins
    elif 'LUC BELAIRE' in desc or 'BELAIRE' in desc:
        return 'Sparkling Blend'
    elif 'VILLA JOLANDA' in desc:
        return 'Sparkling Blend'  # Italian sparkling producer

    # More abbreviation patterns  
    elif 'P/TAGE' in desc:  # Pinotage
        return 'Pinotage'
    elif 'GRUN VELT' in desc or 'GRÜN VELT' in desc:  # Grüner Veltliner
        return 'Grüner Veltliner'
    elif 'GARN' in desc:  # Garnacha
        return 'Garnacha'

    # Rice wine patterns (Korean/Asian)
    elif any(x in desc for x in ['BAEK', 'SOO', 'BOK', 'WHA']):
        return 'Sake'  # Your closest category for rice wines
    
    # More wine name patterns
    elif 'SANT GRIA' in desc or "SANT' GRIA" in desc:
        return 'Sangria'  # Yago makes sangria
    elif 'HONEY WINE' in desc:
        return 'Mead'  # Enat honey wine
    elif 'BERTANI VAL' in desc:
        return 'Valpolicella'  # Bertani Valpolicella
    elif 'WILDLY WICKED' in desc:
        return 'Red Blend'  # Live a Little brand
    
    else:
        return 'Unknown'

# Test the final function
final_results = missing_varieties['ITEM DESCRIPTION'].apply(classify_missing_variety_final)
print("Final classification results:")
print(final_results.value_counts())
print(f"\nTotal classifiable: {(final_results != 'Unknown').sum()}")
print(f"Still unknown: {(final_results == 'Unknown').sum()}")
print(f"Success rate: {(final_results != 'Unknown').sum() / len(final_results) * 100:.1f}%")

In [ ]:
# Step 1: Create the final classification function (copy the big function from earlier)
def classify_missing_variety_final(item_description):
    """
    Final enhanced variety extraction from item descriptions
    """
    if not item_description:
        return 'Unknown'
    
    desc = item_description.upper()
    
    # Direct variety matches
    if 'GRIGIO' in desc or 'P/GRIGIO' in desc:
        return 'Pinot Grigio'
    elif 'CHARDONNAY' in desc or 'CHARD' in desc:
        return 'Chardonnay'
    elif 'CABERNET' in desc:
        if 'SAUVIGNON' in desc:
            return 'Cabernet Sauvignon'
        else:
            return 'Cabernet Sauvignon'
    elif 'MERLOT' in desc:
        return 'Merlot'
    elif 'SAUVIGNON' in desc and 'CABERNET' not in desc:
        return 'Sauvignon Blanc'
    elif 'PINOT' in desc and 'NOIR' in desc:
        return 'Pinot Noir'
    elif 'P/NERO' in desc:  # Italian Pinot Nero
        return 'Pinot Noir'
    elif 'PINOT' in desc:
        return 'Pinot Grigio'
    elif 'SYRAH' in desc:
        return 'Syrah'
    elif 'RIESLING' in desc or 'RIE' in desc:  # Added RIE for German abbreviations
        return 'Riesling'
    elif 'MOSCATO' in desc:
        return 'Moscato'
    elif 'TEMPRANILLO' in desc:
        return 'Tempranillo'
    
    # More variety patterns
    elif 'MALB' in desc:
        return 'Malbec'
    elif 'TORRONTES' in desc:
        return 'Torrontés'
    elif 'GRUNER VELT' in desc or 'GRÜNER VELT' in desc:
        return 'Grüner Veltliner'
    elif 'ALBR' in desc or 'ALBARIÑO' in desc:
        return 'Albariño'
    elif 'PRIMITIVO' in desc:
        return 'Primitivo'
    elif 'SAPERAVI' in desc:
        return 'Saperavi'
    elif 'VIDAL' in desc:
        return 'Vidal Blanc'
    elif 'PICPOUL' in desc:
        return 'Picpoul'
    elif 'NEGRA' in desc:
        return 'Red Blend'
    
    # Regional/style indicators
    elif 'POUILLY FUME' in desc:
        return 'Sauvignon Blanc'
    elif 'MONTEPULCIANO' in desc or 'MONTEPUL' in desc or 'MONT/PUL' in desc:
        return 'Montepulciano'
    elif 'CARMENERE' in desc:
        return 'Carmenère'
    elif 'VERMENTINO' in desc:
        return 'Vermentino'
    elif 'ST EMILION' in desc:
        return 'Red Blend'
    elif 'SUPER TUSCAN' in desc:
        return 'Red Blend'
    elif 'PROVENCE' in desc and ('RSE' in desc or 'ROSE' in desc):
        return 'Rosé'
    elif 'PECORION' in desc:
        return 'Pecorino'
    elif 'ICE' in desc and 'VIDAL' in desc:
        return 'Ice Wine'
    elif 'VALPOLICELLA' in desc or 'RIPASSA VAL' in desc:
        return 'Valpolicella'
    elif 'GSM' in desc:  # Grenache/Syrah/Mourvèdre
        return 'Red Blend'
    
    # NEW: Specific wine name patterns from your list
    elif 'JAZZ BERRY' in desc:
        return 'Fruit Wine'
    elif 'APOTHIC' in desc and 'DARK' in desc:
        return 'Red Blend'
    elif 'CARLO ROSSI PAISANO' in desc:
        return 'Red Blend'
    elif 'SWEET WALTER WHT' in desc:
        return 'White Blend'
    elif 'CHOCOVINE' in desc:
        return 'Red Blend'
    elif 'GOLLY WOBBLER' in desc:
        return 'Red Blend'
    elif 'PECHE IMPERIALE' in desc:
        return 'Moscato'
    elif 'ORIN SWIFT' in desc:
        return 'Red Blend'
    elif 'ANCIANO' in desc and 'RES' in desc:
        return 'Tempranillo'
    elif 'POMEGRANTE' in desc or 'POMEGRANATE' in desc:
        return 'Fruit Wine'
    elif 'PALAZZO DEL TORRE' in desc:
        return 'Red Blend'
    elif 'RES DUCALE' in desc:  # Ruffino Chianti
        return 'Chianti'
    elif 'SEY/CH/VID' in desc:  # Seyval/Chardonnay/Vidal blend
        return 'White Blend'
    
    # Sparkling wines
    elif any(x in desc for x in ['SPARK', 'SPUMANTE', 'ASTI', 'CHAMPAGNE', 
                                 'BL DE NOIR', 'BLANC DE NOIRS', 'CORDON NEGRO',
                                 'ACE OF SPADE']):
        return 'Sparkling Blend'
    elif 'PROSECCO' in desc:
        return 'Prosecco'
    elif 'FREIXENET' in desc:
        return 'Sparkling Blend'
    
    # Fortified wines
    elif any(x in desc for x in ['VERMOUTH', 'VERM', 'SHERRY', 'PORT', 'MADEIRA',
                                 'MARSALA', 'RAINWATER', 'BRISTOL CREAM']):
        return 'Vermouth' if 'VERM' in desc else 'Sherry' if any(x in desc for x in ['SHERRY', 'BRISTOL CREAM']) else 'Madeira' if any(x in desc for x in ['MADEIRA', 'RAINWATER']) else 'Marsala' if 'MARSALA' in desc else 'Port'
    
    # Special categories
    elif 'MEAD' in desc:
        return 'Mead'
    elif 'SAKE' in desc or any(x in desc for x in ['TOZAI', 'KOOK SOON']):
        return 'Sake'
    elif any(x in desc for x in ['MAKKOLI', 'MAKGEOLLI']):
        return 'Sake'  # Korean rice wine, but closest category
    elif 'MD 20/20' in desc:
        return 'Fortified Wine'
    elif 'COLD DUCK' in desc:
        return 'Sparkling Blend'
    elif any(x in desc for x in ['PLUM', 'CHOYA']):
        return 'Fruit Wine'
    
    # Color-based fallbacks
    elif any(word in desc for word in ['RED', 'ROUGE', 'ROSSO', 'TINTO', 'RGE']):
        return 'Red Blend'
    elif any(word in desc for word in ['WHITE', 'BLANC', 'BLANCO', 'BIANCO']):
        return 'White Blend'
    elif any(word in desc for word in ['ROSE', 'RSE', 'ROSÉ']):
        return 'Rosé'

# Quick additions to the classification function:

    # Easy sparkling wins
    elif 'LUC BELAIRE' in desc or 'BELAIRE' in desc:
        return 'Sparkling Blend'
    elif 'VILLA JOLANDA' in desc:
        return 'Sparkling Blend'  # Italian sparkling producer

    # More abbreviation patterns  
    elif 'P/TAGE' in desc:  # Pinotage
        return 'Pinotage'
    elif 'GRUN VELT' in desc or 'GRÜN VELT' in desc:  # Grüner Veltliner
        return 'Grüner Veltliner'
    elif 'GARN' in desc:  # Garnacha
        return 'Garnacha'

    # Rice wine patterns (Korean/Asian)
    elif any(x in desc for x in ['BAEK', 'SOO', 'BOK', 'WHA']):
        return 'Sake'  # Your closest category for rice wines
    
    # More wine name patterns
    elif 'SANT GRIA' in desc or "SANT' GRIA" in desc:
        return 'Sangria'  # Yago makes sangria
    elif 'HONEY WINE' in desc:
        return 'Mead'  # Enat honey wine
    elif 'BERTANI VAL' in desc:
        return 'Valpolicella'  # Bertani Valpolicella
    elif 'WILDLY WICKED' in desc:
        return 'Red Blend'  # Live a Little brand
    
    else:
        return 'Unknown'
    pass

# Step 2: Apply the function to missing varieties only
missing_mask = df['final_variety'] == ''
df.loc[missing_mask, 'final_variety'] = df.loc[missing_mask, 'ITEM DESCRIPTION'].apply(classify_missing_variety_final)

# Step 3: Verify the results
print("Updated variety counts:")
print(df['final_variety'].value_counts().head(20))

print(f"\nBefore: {missing_mask.sum()} missing varieties")
print(f"After: {(df['final_variety'] == 'Unknown').sum()} unknown varieties") 
print(f"Still empty: {(df['final_variety'] == '').sum()} empty varieties")

# Step 4: Optional - convert 'Unknown' to empty string if you prefer
# df.loc[df['final_variety'] == 'Unknown', 'final_variety'] = ''

In [ ]:
# See what variety-related columns exist
variety_cols = [col for col in df.columns if 'variety' in col.lower()]
print("Available variety columns:", variety_cols)

In [ ]:
def classify_wine_color(variety):
    """
    Classify wine variety into color categories
    """
    if not variety or variety.strip() == '':
        return 'Unknown'
    
    variety = variety.strip().lower()
    
    # Red wines
    red_varieties = {
        'red blend', 'cabernet sauvignon', 'pinot noir', 'merlot', 'malbec', 
        'syrah', 'zinfandel', 'tempranillo', 'sangiovese', 'nebbiolo',
        'cabernet franc', 'montepulciano', 'chianti', 'gamay', 'barbera',
        'garnacha', 'portuguese red', 'shiraz', 'nero d\'avola', 'petite sirah',
        'monastrell', 'amarone', 'primitivo', 'pinotage', 'corvina, rondinella, molinara',
        'saperavi', 'nerello mascalese', 'carmenère', 'grenache', 'syrah-viognier',
        'bonarda', 'dolcetto', 'mencía', 'tannat-cabernet', 'cannonau',
        'valpolicella', 'agiorgitiko', 'carignan', 'sagrantino', 'tannat',
        'negroamaro', 'frappato', 'xinomavro', 'malbec-merlot', 'valdiguié',
        'blaufränkisch', 'malvasia nera', 'touriga nacional', 'gamza',
        'cabernet sauvignon-merlot', 'tempranillo-merlot', 'petit verdot',
        'st. laurent', 'teroldego', 'sousão', 'graciano', 'lagrein',
        'chambourcin', 'plavac mali', 'portuguiser', 'monica', 'bobal',
        'mavrud', 'corvina', 'syrah-cabernet', 'feteasca neagra', 'refosco',
        'lemberger', 'pinot noir-gamay', 'syrah-cabernet sauvignon', 
        'dornfelder', 'garnacha tintorera', 'cabernet franc-cabernet sauvignon',
        'piedirosso', 'vranec', 'kekfrankos', 'gaglioppo', 'alicante bouschet',
        'duras', 'schiava', 'argaman', 'prieto picudo', 'papaskarasi',
        'roviello', 'cinsault', 'mandilaria', 'melnik', 'monastrell-syrah',
        'susumaniello'
    }
    
    # White wines
    white_varieties = {
        'chardonnay', 'sauvignon blanc', 'pinot grigio', 'white blend',
        'moscato', 'riesling', 'albariño', 'viognier', 'portuguese white',
        'gewürztraminer', 'chenin blanc', 'garganega', 'verdejo', 'catarratto',
        'viura', 'vinho verde', 'grillo', 'grüner veltliner', 'cortese',
        'torrontés', 'carricante', 'melon', 'moschofilero', 'picpoul',
        'ugni blanc-colombard', 'greco', 'falanghina', 'rkatsiteli',
        'vermentino', 'friulano', 'muscadet', 'assyrtiko', 'sémillon',
        'malvasia', 'savatiano', 'godello', 'verdicchio', 'verdejo-viura',
        'german white blend', 'inzolia', 'vernaccia', 'torbato', 'arneis',
        'rieslaner', 'furmint', 'avesso', 'kisi', 'silvaner', 'pecorino',
        'petit manseng', 'ribolla gialla', 'müller-thurgau', 'grechetto',
        'symphony', 'robola', 'viognier-chardonnay', 'trebbiano', 'turbiana',
        'zibibbo', 'traminette', 'roussanne', 'auxerrois', 'erbaluce',
        'passerina', 'soave', 'kerner', 'vidal blanc', 'greco bianco',
        'antão vaz', 'malvasia istriana', 'chinuri', 'žilavka', 'mtsvane',
        'scheurebe', 'malagousia', 'chenin blanc-chardonnay', 'albana',
        'loureiro', 'jacquère', 'vilana', 'marsanne-viognier', 'picapoll',
        'garnacha blanca', 'malvasia bianca', 'chardonnay-sauvignon',
        'seyval blanc', 'emir', 'tocai'
    }
    
    # Sparkling (can be red, white, or rosé but handled separately)
    sparkling_varieties = {
        'sparkling blend', 'champagne blend', 'prosecco', 'glera',
        'portuguese sparkling', 'lambrusco', 'lambrusco di sorbara',
        'lambrusco grasparossa', 'brachetto'
    }
    
    # Rosé (including white zinfandel)
    rose_varieties = {
        'rosé', 'white zinfandel'
    }
    
    # Fortified/Dessert wines
    fortified_varieties = {
        'port', 'sherry', 'vermouth', 'madeira', 'ice wine', 'mavrodaphne',
        'malaga', 'dessert wine', 'tokaji', 'white port', 'pedro ximénez',
        'terrantez', 'bual', 'palomino', 'fortified wine'
    }
    
    # Other/Special categories
    other_varieties = {
        'sake', 'fruit wine', 'sangria', 'concord'
    }
    
    # Classification logic
    if variety in red_varieties:
        return 'Red'
    elif variety in white_varieties:
        return 'White'
    elif variety in sparkling_varieties:
        return 'Sparkling'
    elif variety in rose_varieties:
        return 'Rosé'
    elif variety in fortified_varieties:
        return 'Fortified'
    elif variety in other_varieties:
        return 'Other'
    else:
        return 'Unknown'
    
# Apply it
df['wine_color'] = df['final_variety'].apply(classify_wine_color)

print("Wine color distribution:")
print(df['wine_color'].value_counts())

In [ ]:
#uncomment below lines to change dataframe.
#df=Warehouse_and_Retail_Sales
#df=Distributors_Virginia_Three_Main
#df=Wine_Review_Data
#df=df_clean_Warehouse_and_Retail_Sales_enhanced
#df=Suppliers_Fixed
#df=df_loaded
#df=df_final
#df=variety_counts

#uncomment below lines to change report
#df.head(25)
df.info()
#df.describe()
#df[df['final_variety'] == ''].head()
#pd.set_option('display.max_rows', None)
#df['ITEM DESCRIPTION'].value_counts()
#deu.investigate_missing_values_manually(df)

In [ ]:
# Save as pickle (recommended for data analysis)
df.to_pickle('wine_data_fully_classified.pkl')

# To load later:
#df = pd.read_pickle('wine_data_fully_classified.pkl')